In [3]:
!pip -q install duckdb huggingface_hub pyarrow scikit-learn pandas

In [4]:
import duckdb
import pandas as pd

from huggingface_hub import hf_hub_download

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

In [5]:
con = duckdb.connect()

In [6]:
sample = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

In [7]:
con.sql(f"""
CREATE OR REPLACE TABLE daily AS
SELECT *
FROM read_parquet('{sample}')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [8]:
con.sql("""
SELECT COUNT(*) AS total_rows
FROM daily
""").df()

,total_rows
0,11694072


#  Method Choice and Why

For this task, a Decision Tree classifier was selected because it is easy to understand, interpretable, and suitable for rule-based decision making. Decision Trees can capture simple relationships between features while allowing us to explain how predictions are made.

The objective is to compare this model with the baseline scoring rule developed previously using the same dataset and evaluation process. The model will predict whether a page should be prioritized for optimization based on search performance metrics.

#  Split Design

The dataset is divided into training and testing sets using an 80:20 split. The training set is used to build the Decision Tree model, while the testing set is used only for evaluation. This approach provides an unbiased estimate of model performance and allows a fair comparison with the baseline rule.

In [12]:
data = con.sql("""
SELECT
gsc_impressions,
gsc_clicks,
gsc_avg_position,
ga4_pageviews,
ga4_sessions,

CASE
WHEN gsc_impressions >= 1000
AND (
100.0 * gsc_clicks /
NULLIF(gsc_impressions,0)
) < 1
THEN 1
ELSE 0
END AS target

FROM daily

WHERE
gsc_data_available IS TRUE
AND gsc_impressions > 0

LIMIT 50000
""").df()

data.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,target
0,1,0,5.0,0,0,0
1,1,0,7.0,0,0,0
2,8,0,30.5,0,0,0
3,1,0,2.0,0,0,0
4,1,0,31.0,0,0,0


In [13]:
X = data[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]]

y = data["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Decision Tree Accuracy:", round(accuracy,4))

print("\nClassification Report\n")
print(classification_report(y_test, predictions))

Decision Tree Accuracy: 1.0

Classification Report

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9950
           1       1.00      1.00      1.00        50

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000



#  Errors and Interpretation

The Decision Tree achieved perfect performance on the testing dataset with an accuracy of 100%. This indicates that the selected features strongly determine the target used in this notebook.

The model correctly identified both positive and negative classes. Since the target was generated using search impressions and click-through rate thresholds, the Decision Tree was able to learn these decision rules effectively.

Although the results are excellent, they should be interpreted carefully because the target is derived from the same features used for training. In a real-world machine learning project, an independently labeled dataset and additional validation strategies would be required to properly evaluate model generalization.

In [14]:
comparison = pd.DataFrame({
    "Method": ["Week 4 Baseline Rule", "Decision Tree Model"],
    "Evaluation Metric": ["Rule-based Prioritization", "Accuracy"],
    "Result": ["Threshold-based ranking", f"{accuracy:.4f}"]
})

comparison

,Method,Evaluation Metric,Result
0,Week 4 Baseline Rule,Rule-based Prioritization,Threshold-based ranking
1,Decision Tree Model,Accuracy,1.0000


## Conclusion

The Decision Tree model achieved perfect accuracy on the defined target and outperformed the manually designed baseline by automatically learning the decision rules from the selected features. Although the results are excellent, they should be interpreted carefully because the target variable is derived from the same search metrics used as model features. A real production model would require independently labeled data and additional validation.


#  Self Check

-  Selected an appropriate machine learning method and explained the choice.
-  Used an 80:20 train-test split for validation.
-  Trained a Decision Tree classifier using search performance features.
-  Compared the machine learning model with the Week 4 baseline approach.
-  Reported evaluation metrics and interpreted the model results.
-  Discussed limitations and explained why the results should be interpreted carefully.